<a href="https://colab.research.google.com/github/mohak1206/signature-detection-/blob/main/Signature_No_signtaure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/signature_yolov8', exist_ok=True)
print("✅ Drive mounted and project folder ready")

Mounted at /content/drive
✅ Drive mounted and project folder ready


In [ ]:
!pip install roboflow ultralytics -q
print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.7 MB/s eta 0:00:00
✅ Dependencies installed


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="sTTCuKGaPL6SEFbsLD7x")
project = rf.workspace("space-3jvwp").project("signature-no-signature")
version = project.version(1)
dataset = version.download("yolov8")

print("✅ Dataset downloaded")
print("Dataset location:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Signature-no-signature-1 in yolov8:: 100%|██████████| 14013/14013 [00:05<00:00, 2488.73it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset downloaded
Dataset location: /content/Signature-no-signature-1


In [ ]:
import os
import shutil
import random
from pathlib import Path

# Collect ALL images and labels from all splits
dataset_path = Path(dataset.location)
all_images = []

for split in ['train', 'valid', 'test']:
    img_dir = dataset_path / split / 'images'
    if img_dir.exists():
        all_images += list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))

print(f"Total images found: {len(all_images)}")

# Shuffle
random.seed(42)
random.shuffle(all_images)

# Calculate splits
total = len(all_images)
train_end = int(0.70 * total)
val_end   = int(0.90 * total)  # 70+20

train_imgs = all_images[:train_end]
val_imgs   = all_images[train_end:val_end]
test_imgs  = all_images[val_end:]

print(f"New Train : {len(train_imgs)} ({len(train_imgs)/total*100:.1f}%)")
print(f"New Val   : {len(val_imgs)}   ({len(val_imgs)/total*100:.1f}%)")
print(f"New Test  : {len(test_imgs)}  ({len(test_imgs)/total*100:.1f}%)")

# Create new split folders
new_base = Path('/content/drive/MyDrive/signature_yolov8/dataset_split')

for split in ['train', 'valid', 'test']:
    (new_base / split / 'images').mkdir(parents=True, exist_ok=True)
    (new_base / split / 'labels').mkdir(parents=True, exist_ok=True)

def copy_split(img_list, split_name):
    for img_path in img_list:
        # Copy image
        shutil.copy(img_path, new_base / split_name / 'images' / img_path.name)
        # Copy matching label
        label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
        if label_path.exists():
            shutil.copy(label_path, new_base / split_name / 'labels' / label_path.name)

copy_split(train_imgs, 'train')
copy_split(val_imgs,   'valid')
copy_split(test_imgs,  'test')

print("✅ Dataset resplit and saved to Drive!")

Total images found: 7004
New Train : 4902 (70.0%)
New Val   : 1401   (20.0%)
New Test  : 701  (10.0%)
✅ Dataset resplit and saved to Drive!


In [ ]:
import yaml

yaml_path = str(new_base / 'data.yaml')

# Read original yaml for class info
orig_yaml = dataset.location + '/data.yaml'
with open(orig_yaml, 'r') as f:
    data = yaml.safe_load(f)

# Update paths to new split
data['train'] = str(new_base / 'train' / 'images')
data['val']   = str(new_base / 'valid' / 'images')
data['test']  = str(new_base / 'test'  / 'images')

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml updated:")
print(f"  Classes : {data['nc']}")
print(f"  Names   : {data['names']}")
print(f"  Train   : {data['train']}")
print(f"  Val     : {data['val']}")
print(f"  Test    : {data['test']}")

✅ data.yaml updated:
  Classes : 2
  Names   : ['rugwed neev signature presence - vdataset rugwed-neev-signature-presence', 'signature']
  Train   : /content/drive/MyDrive/signature_yolov8/dataset_split/train/images
  Val     : /content/drive/MyDrive/signature_yolov8/dataset_split/valid/images
  Test    : /content/drive/MyDrive/signature_yolov8/dataset_split/test/images


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ⚠️")

GPU available: True
Device: Tesla T4


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano = fast; change to yolov8s.pt for better accuracy

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=8,
    name='signature_v1',
    project='/content/drive/MyDrive/signature_yolov8/runs',
    exist_ok=True,
    patience=10,       # early stopping
    save=True,
    plots=True,
    verbose=True
)

print("✅ Training complete! Saved to Drive.")

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/signature_yolov8/dataset_split/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=signature_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer

KeyboardInterrupt: 

In [ ]:
model = YOLO('yolov8m.pt')  # medium — much better accuracy, needs more GPU

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=8,  # increase if GPU allows
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/signature_yolov8/dataset_split/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto,

In [ ]:
from google.colab import files

files.download('/content/runs/detect/train/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
metrics = model.val()

print("✅ Validation Results:")
print(f"  mAP50      : {metrics.box.map50:.4f}")
print(f"  mAP50-95   : {metrics.box.map:.4f}")
print(f"  Precision  : {metrics.box.mp:.4f}")
print(f"  Recall     : {metrics.box.mr:.4f}")

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.3 ms, read: 36.8±23.2 MB/s, size: 80.3 KB)
val: Scanning /content/drive/MyDrive/signature_yolov8/dataset_split/valid/labels.cache... 1401 images, 698 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1401/1401 419.7Mit/s 0.0s
val: /content/drive/MyDrive/signature_yolov8/dataset_split/valid/images/cdp9aa00-page02_1_png_png.rf.ad05fb3842041c3d50d1a571b72fd30d.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/signature_yolov8/dataset_split/valid/images/dlk65e00_png_png.rf.576ad9fde08a23cc1a70fd95c0094343.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/signature_yolov8/dataset_split/valid/images/eld41a00_png_png.rf.1a575fb07d8ecdf02411b1bed4547117.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/signature_yolov8/dataset_split/valid/images/eld41a00_png_png.rf.522f897

In [ ]:
test_results = model.val(
    data=yaml_path,
    split='test'
)

print("✅ Test Set Results:")
print(f"  mAP50      : {test_results.box.map50:.4f}")
print(f"  mAP50-95   : {test_results.box.map:.4f}")
print(f"  Precision  : {test_results.box.mp:.4f}")
print(f"  Recall     : {test_results.box.mr:.4f}")

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 22.3±11.2 MB/s, size: 79.3 KB)
val: Scanning /content/drive/MyDrive/signature_yolov8/dataset_split/test/labels... 701 images, 368 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 701/701 89.0it/s 7.9s
val: /content/drive/MyDrive/signature_yolov8/dataset_split/test/images/gsa_LAR16866-Lease-SF2_Z-02_png_png.rf.6d0d4f4d4eafeea13ec64d5161fe3e1d.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/signature_yolov8/dataset_split/test/images/qit05f00-page2_2_png_png.rf.0a924063c14fddd4856caf1fdecae2a9.jpg: 1 duplicate labels removed
val: New cache created: /content/drive/MyDrive/signature_yolov8/dataset_split/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 464, len(boxes) = 507. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, no